# API Extraction Notebook — Nager.Date Holiday API Brazil

Notebook ini digunakan sebagai lampiran script pengambilan data API untuk Tugas Besar UAS Big Data.

Dataset API yang digunakan adalah **Nager.Date Public Holiday API** untuk mengambil data hari libur publik Brazil tahun 2016–2018.

Notebook ini konsisten dengan pipeline ETL dan ELT utama, yaitu menggunakan data holiday sebagai enrichment untuk fitur `is_public_holiday` dan `holiday_name`.


## 1. Import Library dan Setup Path

Path project mengikuti struktur folder pada notebook ETL dan ELT utama.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import requests
import pandas as pd
from datetime import datetime

PROJECT_PATH = '/content/drive/MyDrive/bigdata_final_project'
RAW_HOLIDAY_PATH = f'{PROJECT_PATH}/raw/holidays'
DATALAKE_RAW_PATH = f'{PROJECT_PATH}/datalake/raw_zone'

os.makedirs(RAW_HOLIDAY_PATH, exist_ok=True)
os.makedirs(DATALAKE_RAW_PATH, exist_ok=True)

print('Project path:', PROJECT_PATH)
print('Raw holiday path:', RAW_HOLIDAY_PATH)


## 2. Extract Data dari Nager.Date API

API endpoint yang digunakan:

```text
https://date.nager.at/api/v3/PublicHolidays/{year}/BR
```

Tahun yang digunakan adalah 2016, 2017, dan 2018 karena periode transaksi Olist berada pada rentang tersebut.


In [ ]:
def extract_nager_holiday_api(years=None, country_code='BR'):
    if years is None:
        years = [2016, 2017, 2018]

    start_time = time.time()
    records = []
    extraction_log = []

    for year in years:
        url = f'https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}'
        response = requests.get(url)
        response.raise_for_status()

        year_data = response.json()
        records.extend(year_data)

        extraction_log.append({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'source': 'Nager.Date Public Holiday API',
            'country_code': country_code,
            'year': year,
            'endpoint': url,
            'rows': len(year_data),
            'status': 'success'
        })

        print(f'Extracted {len(year_data)} holiday records from {url}')

    holiday_df = pd.DataFrame(records)
    execution_time = round(time.time() - start_time, 3)

    print('API extraction finished.')
    print('Total rows:', holiday_df.shape[0])
    print('Total columns:', holiday_df.shape[1])
    print('Execution time:', execution_time, 'seconds')

    return holiday_df, pd.DataFrame(extraction_log)

holiday_df, api_extraction_log = extract_nager_holiday_api()
holiday_df.head()


## 3. Simpan Hasil Extract API ke Raw Folder dan Data Lake

Hasil API disimpan sebagai CSV agar dapat digunakan ulang oleh pipeline ETL dan ELT.


In [ ]:
raw_output_file = f'{RAW_HOLIDAY_PATH}/brazil_holidays_2016_2018.csv'
elt_raw_output_file = f'{RAW_HOLIDAY_PATH}/brazil_holidays_2016_2018_elt_raw.csv'
datalake_output_file = f'{DATALAKE_RAW_PATH}/brazil_holidays_2016_2018.csv'
log_output_file = f'{RAW_HOLIDAY_PATH}/holiday_api_extraction_log.csv'

holiday_df.to_csv(raw_output_file, index=False)
holiday_df.to_csv(elt_raw_output_file, index=False)
holiday_df.to_csv(datalake_output_file, index=False)
api_extraction_log.to_csv(log_output_file, index=False)

print('Holiday raw CSV saved to:', raw_output_file)
print('Holiday ELT raw CSV saved to:', elt_raw_output_file)
print('Holiday datalake raw CSV saved to:', datalake_output_file)
print('API extraction log saved to:', log_output_file)


## 4. Validasi Ringkas Hasil Extract API

Validasi dilakukan untuk memastikan data API berhasil diambil dan memiliki kolom utama yang dibutuhkan untuk proses enrichment.


In [ ]:
required_columns = ['date', 'localName', 'name', 'countryCode']
validation_results = []

validation_results.append({
    'rule_name': 'row_count_greater_than_zero',
    'passed': holiday_df.shape[0] > 0,
    'details': f'total rows: {holiday_df.shape[0]}'
})

missing_columns = [col for col in required_columns if col not in holiday_df.columns]
validation_results.append({
    'rule_name': 'required_columns_exist',
    'passed': len(missing_columns) == 0,
    'details': f'missing columns: {missing_columns}'
})

validation_results.append({
    'rule_name': 'country_code_check',
    'passed': set(holiday_df['countryCode'].dropna().unique()) == {'BR'},
    'details': f'country codes: {holiday_df["countryCode"].dropna().unique().tolist()}'
})

api_validation_df = pd.DataFrame(validation_results)
api_validation_df


## 5. Penjelasan Integrasi dengan Dataset Olist

Data hasil API digunakan sebagai enrichment pada dataset Olist. Proses integrasi dilakukan dengan mencocokkan tanggal order pada Olist dengan tanggal holiday dari Nager.Date API.

Kolom yang digunakan:

```text
Olist: order_date
Holiday API: date
```

Jika `order_date` ditemukan pada data holiday, maka `is_public_holiday = 1`. Jika tidak ditemukan, maka `is_public_holiday = 0`.

Fitur ini digunakan pada pipeline ETL/ELT dan dashboard untuk menganalisis perbandingan revenue pada holiday dan non-holiday.
